In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve, precision_score, recall_score, f1_score, confusion_matrix

file_path = "Customer_Final_Scores_with_Calibration_FIXED.xlsx"
df_raw = pd.read_excel(file_path, sheet_name="RawData")
df_decile = pd.read_excel(file_path, sheet_name="DecileCalibration")

print(df_raw["Sample"].value_counts())


Sample
Train    70
Test     30
Name: count, dtype: int64


In [2]:
# Restrict performance evaluation to the held-out TEST rows only

test_df = df_raw[df_raw["Sample"] == "Test"].copy()

y_true = test_df["Default_y"]
# Flip the score so higher predicted-risk = higher y_score (needed for roc_auc_score)
y_score = -test_df["Final_Score"]

print(f"Evaluating on {len(test_df)} held-out test customers "
      f"({test_df['Default_y'].sum()} bads, {(test_df['Default_y']==0).sum()} goods)")

Evaluating on 30 held-out test customers (9 bads, 21 goods)


In [3]:
# 1. Gini & AR (test set only)
auc = roc_auc_score(y_true, y_score)
gini = 2 * auc - 1
ar = gini / 2

print(f"AUC (test): {auc:.3f}")
print(f"Gini (test): {gini:.3f}")
print(f"Accuracy Ratio (AR, test): {ar:.3f}")


AUC (test): 0.429
Gini (test): -0.143
Accuracy Ratio (AR, test): -0.071


In [4]:
# 2. KS Statistic (test set only)
fpr, tpr, thresholds = roc_curve(y_true, y_score)
ks_stat = max(tpr - fpr)
print(f"KS Statistic (test): {ks_stat:.3f}")

KS Statistic (test): 0.127


In [5]:
# 3. Rank Order (test set only)
# With a small test set, use fewer buckets (5) so each bucket has enough customers
# to give a meaningful bad rate. Drop to fewer still if you see empty/ties warnings.
n_buckets = 5
test_df["score_bucket"] = pd.qcut(y_score, n_buckets, labels=False, duplicates="drop")

rank_order = test_df.groupby("score_bucket").agg(
    total=("Default_y", "count"),
    bads=("Default_y", "sum"),
    goods=("Default_y", lambda x: (x == 0).sum()),
    bad_rate=("Default_y", "mean"),
    avg_score=("Final_Score", "mean")
).reset_index()

print("Rank Order Table (test set, should show bad_rate roughly increasing "
      "as avg_score decreases):")
print(rank_order)

Rank Order Table (test set, should show bad_rate roughly increasing as avg_score decreases):
   score_bucket  total  bads  goods  bad_rate   avg_score
0             0      6     1      5  0.166667  878.333333
1             1      6     3      3  0.500000  752.666667
2             2      6     2      4  0.333333  662.166667
3             3      6     2      4  0.333333  562.333333
4             4      6     1      5  0.166667  414.000000


In [6]:
# 4. PSI: development (Train) score distribution vs. evaluation (Test) score distribution
def calculate_psi(expected, actual, buckets=10):
    breakpoints = np.percentile(expected, np.arange(0, buckets + 1) / buckets * 100)
    breakpoints = np.unique(breakpoints)

    expected_percents = np.histogram(expected, breakpoints)[0] / len(expected)
    actual_percents = np.histogram(actual, breakpoints)[0] / len(actual)

    expected_percents = np.where(expected_percents == 0, 1e-6, expected_percents)
    actual_percents = np.where(actual_percents == 0, 1e-6, actual_percents)

    return np.sum((actual_percents - expected_percents) *
                  np.log(actual_percents / expected_percents))

train_scores = df_raw.loc[df_raw["Sample"] == "Train", "Final_Score"]
test_scores  = df_raw.loc[df_raw["Sample"] == "Test", "Final_Score"]

# Fewer buckets given the small sample size (30 test rows)
psi_val = calculate_psi(train_scores, test_scores, buckets=5)
print(f"PSI (Train vs Test score distribution): {psi_val:.3f}")
print("Rule of thumb: <0.10 stable, 0.10-0.25 moderate shift, >0.25 significant shift")

PSI (Train vs Test score distribution): 0.611
Rule of thumb: <0.10 stable, 0.10-0.25 moderate shift, >0.25 significant shift


In [7]:
# 5. Score Concentration (test set)
score_threshold = np.percentile(test_df["Final_Score"], 80)
concentration = (test_df["Final_Score"] >= score_threshold).mean()
print(f"Score Concentration (Top 20% customers, test set): {concentration:.2%}")

Score Concentration (Top 20% customers, test set): 20.00%


In [8]:
# 6. Concordance / Discordance (test set)
def concordance_discordance(y_true, y_score):
    good_scores = y_score[y_true == 0]
    bad_scores = y_score[y_true == 1]
    total_pairs = len(good_scores) * len(bad_scores)

    concordant = discordant = ties = 0
    for g in good_scores:
        concordant += np.sum(g > bad_scores)
        discordant += np.sum(g < bad_scores)
        ties += np.sum(g == bad_scores)

    return {
        "concordant_pct": concordant / total_pairs,
        "discordant_pct": discordant / total_pairs,
        "ties_pct": ties / total_pairs,
    }

conc_disc = concordance_discordance(y_true.values, y_score.values)
print("Concordance / Discordance (test set):")
print(conc_disc)


Concordance / Discordance (test set):
{'concordant_pct': np.float64(0.5714285714285714), 'discordant_pct': np.float64(0.42857142857142855), 'ties_pct': np.float64(0.0)}


In [9]:
#Precision / Recall / F1 at a matched approval-rate cutoff (Test set)
#Bottom 20% of scores = declined (highest-risk 20% of applicants), i.e. an 80% approval rate.
#Using the same 20% reject rate here as in the champion-vs-challenger comparison (Step 9)
#keeps every precision/recall/F1 figure in this project comparable on equal footing.
cutoff_score = np.percentile(test_df["Final_Score"], 20)  # bottom 20% of scores = highest risk
y_pred = (test_df["Final_Score"] < cutoff_score).astype(int)  # 1 = predicted bad

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print(f"Cutoff Score: {cutoff_score:.1f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
print("\nConfusion Matrix [ [TN, FP], [FN, TP] ]:")
print(cm)

Cutoff Score: 529.4
Precision: 0.167
Recall: 0.111
F1 Score: 0.133

Confusion Matrix [ [TN, FP], [FN, TP] ]:
[[16  5]
 [ 8  1]]


In [10]:
#CSI (Characteristic Stability Index)
#Computed across every risk variable that feeds the model (26 predictors from
#Final_Model_Dataset.xlsx: bureau, application, and behavioral variables + the
#segmentation Cluster), not just a handful — this is what lets CSI actually
#support a stability conclusion at the full-variable-set level expected in
#model governance validation.

def calculate_csi(expected, actual, buckets=10):
    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()

    breakpoints = np.percentile(expected, np.arange(0, buckets + 1) / buckets * 100)
    breakpoints = np.unique(breakpoints)

    expected_percents = np.histogram(expected, breakpoints)[0] / len(expected)
    actual_percents = np.histogram(actual, breakpoints)[0] / len(actual)

    expected_percents = np.where(expected_percents == 0, 1e-6, expected_percents)
    actual_percents = np.where(actual_percents == 0, 1e-6, actual_percents)

    return np.sum((actual_percents - expected_percents) * np.log(actual_percents / expected_percents))

dev_sample = df_raw[df_raw["Sample"] == "Train"]
current_sample = df_raw[df_raw["Sample"] == "Test"]

# Non-risk-variable columns to exclude: target, derived score/sample fields
exclude_cols = {"Default_y", "Final_Score", "Sample", "score_decile", "score_bucket",
                 "LR_OOF_Proba", "XGB_OOF_Proba"}

characteristics = [
    c for c in df_raw.columns
    if c not in exclude_cols and pd.api.types.is_numeric_dtype(df_raw[c])
]

print(f"Running CSI across {len(characteristics)} risk variables (Train vs Test):")
print(characteristics)

csi_results = []
for col in characteristics:
    csi_val = calculate_csi(dev_sample[col], current_sample[col], buckets=5)
    flag = "Stable" if csi_val < 0.1 else ("Moderate Shift" if csi_val < 0.25 else "Significant Shift")
    csi_results.append({"characteristic": col, "csi": round(csi_val, 4), "flag": flag})

csi_df = pd.DataFrame(csi_results).sort_values("csi", ascending=False)
print("\nCharacteristic Stability Index (CSI) by variable, Train vs Test:")
print(csi_df.to_string(index=False))
print(f"\nVariables flagged Moderate/Significant Shift: {(csi_df['flag'] != 'Stable').sum()} of {len(csi_df)}")

Running CSI across 26 risk variables (Train vs Test):
['Behavior_Spending_Score', 'Income_INR', 'Total_Credit_Limit', 'Loan_Tenure_Months', 'Months_Since_Most_Recent_Delinquency', 'Credit_Utilization_Ratio', 'Credit_Card_Utilization', 'Employment_Years', 'Behavior_Repayment_Score', 'No_of_Inquiries_6M', 'Credit_History_Length', 'Worst_Current_Status', 'No_of_Closed_Accounts', 'Newest_Trade_Open_Months', 'Checking_Account_Balance', 'No_of_Inquiries_12M', 'Delinquency_12M', 'Loan_Amount', 'Outstanding_Loans', 'Oldest_Trade_Open_Months', 'Total_Current_Balance', 'Max_Credit_Exposure', 'No_of_Open_Accounts', 'DPD_30', 'Savings_Account_Balance', 'Cluster']

Characteristic Stability Index (CSI) by variable, Train vs Test:
                      characteristic    csi              flag
               Total_Current_Balance 0.4682 Significant Shift
                 No_of_Inquiries_12M 0.4323 Significant Shift
                  Total_Credit_Limit 0.3863 Significant Shift
                          